# Notebook 12 — Pipeline RAG

---
## 1. Importaciones

Se importan las funciones del módulo `rag_utils`, que centraliza toda la lógica del pipeline.

In [1]:
import sys, os, time, json
import numpy as np
import pandas as pd
from collections import defaultdict

sys.path.append(os.path.abspath('../../../..'))

from src.Proyecto_3.rag_utils import (
    cargar_desde_csv,
    limpiar_texto,
    chunk_por_cancion,
    chunk_por_estrofa,
    construir_indice,
    buscar_chunks,
    stats_corpus,
    _get_model,
    EMBED_CACHE,
    CHUNKS_CACHE,
)
import faiss

print('✓ Importaciones completadas')

D:\kenda\Downloads\Analisis-Morfosintactico-de-Letras-Musicales-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Importaciones completadas


---
## 2. Carga y limpieza del corpus

In [2]:
df = cargar_desde_csv()

print(f'\nEstadísticas del corpus:')
stats = stats_corpus(df)
for genero, n in stats.items():
    print(f'  {genero}: {n} canciones')

print(f'\nEjemplo de lyric limpia (primeros 300 chars):')
print(df['Lyrics'].iloc[0][:300])

Leyendo CSV desde: D:\kenda\Downloads\Analisis-Morfosintactico-de-Letras-Musicales-main (3)\Analisis-Morfosintactico-de-Letras-Musicales-main\src\Proyecto_3\..\..\data\raw\spotify_dataset.csv
  → 290183 filas totales en el CSV
  → Columnas detectadas: ['Artist', 'Song', 'Genre', 'Language', 'Lyrics']
  → Géneros únicos detectados: ['Rock', 'Metal', 'Pop', 'Indie', 'R&B', 'Folk', 'Electronic', 'Jazz', 'Hip-Hop', 'Country']
  → 143935 canciones tras filtrar géneros: ['Rock', 'Hip-Hop', 'Metal']
  → 120393 canciones únicas (eliminados 23542 duplicados)
  → 119249 canciones con lyrics válidas (descartadas 1144 demasiado cortas/vacías)
  → Dataset balanceado: 2225 canciones por género (seed=42)
  → Conteo final por género:
Genre
Hip-Hop    2225
Metal      2225
Rock       2225
  → Total: 6675 canciones
✓ CSV procesado guardado en: D:\kenda\Downloads\Analisis-Morfosintactico-de-Letras-Musicales-main (3)\Analisis-Morfosintactico-de-Letras-Musicales-main\src\Proyecto_3\..\..\data\processed\canc

---
## 3. Estrategias de chunking

El chunking define cómo se fragmenta el corpus para la búsqueda semántica. Dos estrategias muy distintas en granularidad:

### 3.1 — Estrategia 1: Canción completa

In [3]:
chunks_cancion = chunk_por_cancion(df)

lens_cancion = [len(c['texto']) for c in chunks_cancion]
print(f'Estrategia 1 — Por canción completa')
print(f'  Total chunks  : {len(chunks_cancion)}')
print(f'  Longitud media: {np.mean(lens_cancion):.0f} chars')
print(f'  Longitud mín  : {np.min(lens_cancion)} chars')
print(f'  Longitud máx  : {np.max(lens_cancion)} chars')
print(f'  Desv. estándar: {np.std(lens_cancion):.0f} chars')

# Distribución por género
por_genero = defaultdict(int)
for c in chunks_cancion:
    por_genero[c['genre']] += 1
print(f'\n  Chunks por género:')
for g, n in sorted(por_genero.items()):
    print(f'    {g}: {n}')

Estrategia 1 — Por canción completa
  Total chunks  : 6675
  Longitud media: 1098 chars
  Longitud mín  : 50 chars
  Longitud máx  : 1500 chars
  Desv. estándar: 390 chars

  Chunks por género:
    Hip-Hop: 2225
    Metal: 2225
    Rock: 2225


### 3.2 — Estrategia 2: Por estrofa

In [4]:
chunks_estrofa = chunk_por_estrofa(df)

lens_estrofa = [len(c['texto']) for c in chunks_estrofa]
print(f'Estrategia 2 — Por estrofa')
print(f'  Total chunks  : {len(chunks_estrofa)}')
print(f'  Longitud media: {np.mean(lens_estrofa):.0f} chars')
print(f'  Longitud mín  : {np.min(lens_estrofa)} chars')
print(f'  Longitud máx  : {np.max(lens_estrofa)} chars')
print(f'  Desv. estándar: {np.std(lens_estrofa):.0f} chars')

# Promedio de estrofas por canción
estrofas_por_cancion = len(chunks_estrofa) / len(df)
print(f'\n  Promedio estrofas/canción: {estrofas_por_cancion:.1f}')

por_genero_e = defaultdict(int)
for c in chunks_estrofa:
    por_genero_e[c['genre']] += 1
print(f'\n  Chunks por género:')
for g, n in sorted(por_genero_e.items()):
    print(f'    {g}: {n}')

Estrategia 2 — Por estrofa
  Total chunks  : 17232
  Longitud media: 315 chars
  Longitud mín  : 40 chars
  Longitud máx  : 800 chars
  Desv. estándar: 284 chars

  Promedio estrofas/canción: 2.6

  Chunks por género:
    Hip-Hop: 2225
    Metal: 7898
    Rock: 7109


---
## 4. Comparación formal de estrategias

creacion de textos para futura predicion

In [5]:
EVAL_QUERIES = [
    # Rock
    {'query': 'rebellion freedom electric guitar',       'genero_esperado': 'Rock'},
    {'query': 'road trip journey classic rock anthem',   'genero_esperado': 'Rock'},
    {'query': 'heartbreak love lost rock ballad',        'genero_esperado': 'Rock'},
    {'query': 'protest society political rock lyrics',   'genero_esperado': 'Rock'},
    # Hip-Hop
    {'query': 'street life hustle urban grind',          'genero_esperado': 'Hip-Hop'},
    {'query': 'money fame success rap ambition',         'genero_esperado': 'Hip-Hop'},
    {'query': 'poverty inequality social justice rap',   'genero_esperado': 'Hip-Hop'},
    {'query': 'flow rhyme wordplay lyrical skill',       'genero_esperado': 'Hip-Hop'},
    # Metal
    {'query': 'darkness evil demons heavy metal',        'genero_esperado': 'Metal'},
    {'query': 'war battle warrior power metal',          'genero_esperado': 'Metal'},
    {'query': 'death mortality despair metal lyrics',    'genero_esperado': 'Metal'},
    {'query': 'aggression anger rage heavy music',       'genero_esperado': 'Metal'},
]

print(f'Total de queries de evaluación: {len(EVAL_QUERIES)}')
for q in EVAL_QUERIES:
    print(f"  [{q['genero_esperado']}] {q['query']}")

Total de queries de evaluación: 12
  [Rock] rebellion freedom electric guitar
  [Rock] road trip journey classic rock anthem
  [Rock] heartbreak love lost rock ballad
  [Rock] protest society political rock lyrics
  [Hip-Hop] street life hustle urban grind
  [Hip-Hop] money fame success rap ambition
  [Hip-Hop] poverty inequality social justice rap
  [Hip-Hop] flow rhyme wordplay lyrical skill
  [Metal] darkness evil demons heavy metal
  [Metal] war battle warrior power metal
  [Metal] death mortality despair metal lyrics
  [Metal] aggression anger rage heavy music


### 4.2 Construcción de índices para ambas estrategias

Se construyen dos índices FAISS independientes para poder comparar.

In [6]:
import pickle

model = _get_model()

def construir_indice_local(chunks, nombre):
    """Construye un índice FAISS sin usar caché global, para la comparación."""
    print(f'\n⚙ Construyendo índice [{nombre}] con {len(chunks)} chunks...')
    t0 = time.time()
    
    textos = [c['texto'] for c in chunks]
    embeddings = model.encode(textos, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)
    
    dim = embeddings.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(embeddings)
    
    t1 = time.time()
    print(f'  ✓ Índice listo en {t1-t0:.1f}s — {idx.ntotal} vectores de {idx.d} dims')
    return idx, embeddings, t1 - t0


idx_cancion,  emb_cancion,  t_cancion  = construir_indice_local(chunks_cancion,  'cancion_completa')
idx_estrofa,  emb_estrofa,  t_estrofa  = construir_indice_local(chunks_estrofa,  'estrofa')

Cargando modelo de embeddings...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6456.12it/s]



⚙ Construyendo índice [cancion_completa] con 6675 chunks...


Batches: 100%|██████████| 105/105 [03:04<00:00,  1.76s/it]


  ✓ Índice listo en 184.6s — 6675 vectores de 384 dims

⚙ Construyendo índice [estrofa] con 17232 chunks...


Batches: 100%|██████████| 270/270 [04:24<00:00,  1.02it/s]

  ✓ Índice listo en 264.8s — 17232 vectores de 384 dims


### 4.3 Función de evaluación

In [7]:
def evaluar_estrategia(idx, chunks, nombre, queries, top_k=5):
    """
    Evalúa una estrategia de chunking sobre las queries de prueba.
    Retorna un dict con métricas agregadas.[
    """
    scores_mean      = []   # similitud coseno promedio del top-K
    scores_top1      = []   # similitud coseno del resultado #1
    precision_genero = []   # % de resultados en el género esperado
    tiempos_query    = []

    detalles = []

    for item in queries:
        q       = item['query']
        g_esp   = item['genero_esperado']

        t0 = time.time()
        q_emb = model.encode([q], convert_to_numpy=True)
        faiss.normalize_L2(q_emb)
        D, I = idx.search(q_emb, top_k)
        t1 = time.time()

        valid = [(float(D[0][i]), chunks[I[0][i]]) for i in range(top_k) if I[0][i] >= 0]

        if not valid:
            continue

        scs = [v[0] for v in valid]
        gen = [v[1]['genre'] for v in valid]

        prec = sum(1 for g in gen if g.lower() == g_esp.lower()) / len(gen)

        scores_mean.append(np.mean(scs))
        scores_top1.append(scs[0])
        precision_genero.append(prec)
        tiempos_query.append((t1 - t0) * 1000)  # ms

        detalles.append({
            'query':          q,
            'genero_esperado': g_esp,
            'score_top1':     round(scs[0], 4),
            'score_mean':     round(np.mean(scs), 4),
            'precision_genero': round(prec, 4),
            'top5_generos':   gen,
        })

    return {
        'nombre':            nombre,
        'n_chunks':          len(chunks),
        'tiempo_indexado_s': round(t_cancion if nombre == 'cancion_completa' else t_estrofa, 2),
        'score_top1_mean':   round(np.mean(scores_top1), 4),
        'score_topk_mean':   round(np.mean(scores_mean), 4),
        'precision_genero':  round(np.mean(precision_genero), 4),
        'latencia_ms_mean':  round(np.mean(tiempos_query), 2),
        'detalles':          detalles,
    }


print('Evaluando estrategia 1: canción completa...')
res_cancion = evaluar_estrategia(idx_cancion, chunks_cancion, 'cancion_completa', EVAL_QUERIES)

print('Evaluando estrategia 2: estrofa...')
res_estrofa = evaluar_estrategia(idx_estrofa, chunks_estrofa, 'estrofa', EVAL_QUERIES)

print('\n✓ Evaluación completada.')

Evaluando estrategia 1: canción completa...
Evaluando estrategia 2: estrofa...

✓ Evaluación completada.


### 4.4 Tabla comparativa de métricas

In [8]:
metricas_df = pd.DataFrame([
    {
        'Estrategia':           r['nombre'],
        'N° chunks':            r['n_chunks'],
        'Score top-1 (↑)':     r['score_top1_mean'],
        'Score top-K mean (↑)': r['score_topk_mean'],
        'Precisión género (↑)': f"{r['precision_genero']*100:.1f}%",
        'Tiempo indexado (s)':  r['tiempo_indexado_s'],
        'Latencia búsqueda (ms)': r['latencia_ms_mean'],
    }
    for r in [res_cancion, res_estrofa]
])

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('=' * 90)
print('COMPARACIÓN DE ESTRATEGIAS DE CHUNKING')
print('=' * 90)
print(metricas_df.to_string(index=False))
print('=' * 90)
print('(↑) = mayor es mejor')

COMPARACIÓN DE ESTRATEGIAS DE CHUNKING
      Estrategia  N° chunks  Score top-1 (↑)  Score top-K mean (↑) Precisión género (↑)  Tiempo indexado (s)  Latencia búsqueda (ms)
cancion_completa       6675           0.5627                0.5290                50.0%               184.65                   22.60
         estrofa      17232           0.6098                0.5734                53.3%               264.81                   21.26
(↑) = mayor es mejor


### 4.5 Detalle por query

Utilizamos las oraciones previas para comparar datos


In [9]:
print('\n── Detalle por query: CANCIÓN COMPLETA ──')
det_c = pd.DataFrame(res_cancion['detalles'])[
    ['query', 'genero_esperado', 'score_top1', 'score_mean', 'precision_genero']
]
print(det_c.to_string(index=False))

print('\n── Detalle por query: ESTROFA ──')
det_e = pd.DataFrame(res_estrofa['detalles'])[
    ['query', 'genero_esperado', 'score_top1', 'score_mean', 'precision_genero']
]
print(det_e.to_string(index=False))


── Detalle por query: CANCIÓN COMPLETA ──
                                query genero_esperado  score_top1  score_mean  precision_genero
    rebellion freedom electric guitar            Rock      0.5109      0.4712               0.2
road trip journey classic rock anthem            Rock      0.5302      0.4950               0.8
     heartbreak love lost rock ballad            Rock      0.6470      0.6164               0.4
protest society political rock lyrics            Rock      0.5437      0.5318               0.2
       street life hustle urban grind         Hip-Hop      0.4689      0.4574               0.2
      money fame success rap ambition         Hip-Hop      0.5095      0.4939               1.0
poverty inequality social justice rap         Hip-Hop      0.5109      0.4543               0.0
    flow rhyme wordplay lyrical skill         Hip-Hop      0.5776      0.5142               0.2
     darkness evil demons heavy metal           Metal      0.6141      0.6037               1

### 4.6 Selección automática de la mejor estrategia


In [10]:
def puntaje_compuesto(r, w1=0.40, w2=0.30, w3=0.30):
    return w1 * r['score_top1_mean'] + w2 * r['score_topk_mean'] + w3 * r['precision_genero']

p_cancion = puntaje_compuesto(res_cancion)
p_estrofa = puntaje_compuesto(res_estrofa)

print(f'Puntaje compuesto — Canción completa: {p_cancion:.4f}')
print(f'Puntaje compuesto — Estrofa:          {p_estrofa:.4f}')

#Simples comparacion con un if
if p_cancion >= p_estrofa:
    MEJOR_ESTRATEGIA    = 'cancion_completa'
    mejor_chunks        = chunks_cancion
    mejor_idx           = idx_cancion
    MEJOR_NOMBRE        = 'Canción completa'
else:
    MEJOR_ESTRATEGIA    = 'estrofa'
    mejor_chunks        = chunks_estrofa
    mejor_idx           = idx_estrofa
    MEJOR_NOMBRE        = 'Estrofa'

print(f'\n🏆 MEJOR ESTRATEGIA: {MEJOR_NOMBRE} (puntaje = {max(p_cancion, p_estrofa):.4f})')
print(f'   → Se usará esta estrategia para el índice FAISS definitivo del chatbot.')

Puntaje compuesto — Canción completa: 0.5338
Puntaje compuesto — Estrofa:          0.5759

🏆 MEJOR ESTRATEGIA: Estrofa (puntaje = 0.5759)
   → Se usará esta estrategia para el índice FAISS definitivo del chatbot.


---
## 5. Construcción del índice FAISS definitivo

Se construye el índice con la estrategia ganadora y se cachea en disco para que el chatbot no regenere embeddings en cada ejecución.

In [11]:
print(f'Construyendo índice definitivo con estrategia: {MEJOR_NOMBRE}')
index, chunks = construir_indice(chunks=mejor_chunks, forzar=True)

print(f'\n✓ Índice FAISS definitivo: {index.ntotal} vectores de {index.d} dimensiones')
print(f'   Estrategia activa: {MEJOR_ESTRATEGIA}')

Construyendo índice definitivo con estrategia: Estrofa
Generando embeddings para 17232 chunks...


Batches: 100%|██████████| 539/539 [04:18<00:00,  2.09it/s]


Embeddings guardados en caché.
✓ Índice FAISS listo con 17232 vectores.

✓ Índice FAISS definitivo: 17232 vectores de 384 dimensiones
   Estrategia activa: estrofa


---
## 6. Búsqueda semántica

### 6.1 ¿Cómo funciona la búsqueda?

1. La **query** del usuario se convierte en un vector de embedding con el mismo modelo
2. FAISS busca los **top-K vectores** con mayor similitud coseno en el índice
3. Se retornan los chunks con su **score** (entre 0 y 1, donde 1 = idéntico)
4. Un `filtro_genero` opcional restringe los resultados a un género específico

### 6.2 Queries de prueba — búsqueda libre

In [12]:
queries_demo = [
    'songs about rebellion and freedom',
    'dark heavy aggressive lyrics',
    'street life urban poetry',
]

for q in queries_demo:
    print(f'\n🔍 Query: "{q}"')
    resultados = buscar_chunks(q, top_k=3)
    for r in resultados:
        print(f'  [{r["genre"]}] {r["song"]} — {r["artist"]} (score: {r["score"]:.3f})')
        print(f'    Fragmento: {r["texto"][:120]}...')


🔍 Query: "songs about rebellion and freedom"
  [Hip-Hop] freedom now — boom shaka (score: 0.629)
    Fragmento: chorus Freedom now Freedom songs People want their freedom now People need their freedom now People want their freedom n...
  [Metal] the great satan [#] — ministry (score: 0.591)
    Fragmento: The trumpet of freedom has sounded
Great Satan...
  [Hip-Hop] aquarius — common (score: 0.576)
    Fragmento: Yeah Yeah Nigga deep in the rhythm experience speak Some keepin the wisdom the life hustlers seek I seeking it with em i...

🔍 Query: "dark heavy aggressive lyrics"
  [Metal] worlds and machines — vicious rumors (score: 0.651)
    Fragmento: Lyrics Music Vicious Rumors
Everytime I turn around
Someone tries to bring me down
Spinning in a whirlpool
over rules ag...
  [Rock] hunted down — soundgarden (score: 0.630)
    Fragmento: Lyrics by Chris Cornell
Music by Kim Thayil...
  [Rock] spiteful child — elton john (score: 0.573)
    Fragmento: Music by Elton John
Lyrics by Bernie 

---
## 7. Conclusiones

### 7.1 Resumen de la comparación

In [15]:
print('=' * 70)
print('RESUMEN FINAL — COMPARACIÓN DE ESTRATEGIAS DE CHUNKING')
print('=' * 70)

for r, p in [(res_cancion, p_cancion), (res_estrofa, p_estrofa)]:
    ganadora = ' ← GANADORA' if r['nombre'] == MEJOR_ESTRATEGIA else ''
    print(f"\n  Estrategia    : {r['nombre']}{ganadora}")
    print(f"  N° chunks     : {r['n_chunks']}")
    print(f"  Score top-1   : {r['score_top1_mean']:.4f}")
    print(f"  Score top-K   : {r['score_topk_mean']:.4f}")
    print(f"  Prec. género  : {r['precision_genero']*100:.1f}%")
    print(f"  Puntaje comp. : {p:.4f}")

print(f'\n  Estrategia seleccionada para el chatbot: {MEJOR_NOMBRE}')
print('=' * 70)

# Guardar resultado de la comparación
resultado_comparacion = {
    'mejor_estrategia': MEJOR_ESTRATEGIA,
    'puntaje_cancion_completa': round(p_cancion, 6),
    'puntaje_estrofa': round(p_estrofa, 6),
    'metricas_cancion_completa': {
        k: v for k, v in res_cancion.items() if k != 'detalles'
    },
    'metricas_estrofa': {
        k: v for k, v in res_estrofa.items() if k != 'detalles'
    },
}

import os
os.makedirs('resultados', exist_ok=True)
with open('resultados/chunking_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_comparacion, f, ensure_ascii=False, indent=2)

print('\n✓ Resultados guardados en: resultados/chunking_comparison.json')

RESUMEN FINAL — COMPARACIÓN DE ESTRATEGIAS DE CHUNKING

  Estrategia    : cancion_completa
  N° chunks     : 6675
  Score top-1   : 0.5627
  Score top-K   : 0.5290
  Prec. género  : 50.0%
  Puntaje comp. : 0.5338

  Estrategia    : estrofa ← GANADORA
  N° chunks     : 17232
  Score top-1   : 0.6098
  Score top-K   : 0.5734
  Prec. género  : 53.3%
  Puntaje comp. : 0.5759

  Estrategia seleccionada para el chatbot: Estrofa

✓ Resultados guardados en: resultados/chunking_comparison.json
